# 07 — Multi-Objective Pareto Optimizer (BoTorch)

Uses BoTorch to find Pareto-optimal ALD showerhead designs by optimising
three competing objectives:

| Objective | Definition | Direction |
|-----------|-----------|----------|
| **TMA uniformity** | 1 − CV(TMA) at standoff mid-plane | maximise |
| **T uniformity** | 1 − CV(T) at wafer plane | maximise |
| **Pressure drop** | −Eu (Euler number) | maximise (= minimise Eu) |

**Two-level approach:**
1. Run `MultiHeadMGN` surrogate on 80 existing cases → compute scalar objectives
2. Fit BoTorch `SingleTaskGP` over design params **[log D, pitch/D, log Q]** → objectives
3. Run `qLogNEHVI` (multi-objective BO) to find Pareto front in continuous design space
4. Flag top Pareto candidates for full surrogate evaluation + guardrail check

**Design space:**
- D ∈ [1, 3] mm (nozzle diameter)
- pitch/D ∈ [3, 6]
- Q ∈ [1, 10] slm (flow rate)

**Requires:** Colab A100 GPU

## 0. Setup

In [ ]:
import subprocess, sys, os

subprocess.run(['pip', 'install', '-q', 'torch_geometric', 'h5py', 'scipy',
                'botorch', 'gpytorch'], check=True)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/Ranaam21/cfd-ald-app.git'
REPO_DIR = '/content/cfd-ald-app'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(r.stdout.strip() or 'Repo up to date.')
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Cloned.')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Setup done.')

## 1. Paths + imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import h5py
import json
from pathlib import Path
from tqdm import tqdm
from scipy.spatial import cKDTree

from physics.guardrails import GuardrailEngine
from physics.calculator import euler

assert torch.cuda.is_available(), 'Switch to A100 GPU runtime'
DEVICE = torch.device('cuda')
print(f'GPU: {torch.cuda.get_device_name(0)}')

DRIVE_BASE   = Path('/content/drive/MyDrive/cfd-ald-app')
HDF5_DIR     = DRIVE_BASE / 'showerhead_openfoam'
MULTIHEAD_PT = DRIVE_BASE / 'checkpoints' / 'multihead' / 'multihead_final.pt'
OUT_DIR      = DRIVE_BASE / 'checkpoints' / 'optimizer'
OUT_DIR.mkdir(parents=True, exist_ok=True)

h5_files = sorted(HDF5_DIR.glob('case_*.h5'))
print(f'Found {len(h5_files)} HDF5 files')

GLOBAL_KEYS = ['Re','Pr','Sc','Ma','Pe_h','Pe_m','Da',
                'D','pitch_over_D','H_plenum','t_face','standoff',
                'flow_rate_slm','beta','v_th','D_m','n_holes','open_area_frac']
OUTPUT_COLS = ['Ux','Uy','Uz','p','T','TMA']
RHO_N2 = 1.145   # kg/m³ at 300 K

print('Imports OK.')

## 2. Load multihead surrogate

In [ ]:
from torch_geometric.nn import MessagePassing

def mlp(in_dim, out_dim, hidden=None, n_layers=2):
    hidden = hidden or out_dim
    dims = [in_dim] + [hidden] * (n_layers - 1) + [out_dim]
    mods = []
    for i in range(len(dims) - 1):
        mods.append(nn.Linear(dims[i], dims[i+1]))
        if i < len(dims) - 2:
            mods.append(nn.SiLU())
    return nn.Sequential(*mods)

class MGNProcessor(MessagePassing):
    def __init__(self, hidden):
        super().__init__(aggr='sum')
        self.edge_mlp  = mlp(3*hidden, hidden)
        self.node_mlp  = mlp(2*hidden, hidden)
        self.edge_norm = nn.LayerNorm(hidden)
        self.node_norm = nn.LayerNorm(hidden)
    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        N = x.size(0)
        new_e = self.edge_norm(self.edge_mlp(torch.cat([x[src], x[dst], edge_attr], -1)) + edge_attr)
        agg   = self.propagate(edge_index, x=x, edge_attr=new_e, size=(N, N))
        new_x = self.node_norm(self.node_mlp(torch.cat([x, agg], -1)) + x)
        return new_x, new_e
    def message(self, edge_attr): return edge_attr

class MultiHeadMGN(nn.Module):
    def __init__(self, node_dim, edge_dim, flow_out=4, heat_out=1, species_out=1, hidden=128, n_layers=8):
        super().__init__()
        self.node_enc   = mlp(node_dim, hidden)
        self.edge_enc   = mlp(edge_dim, hidden)
        self.processors = nn.ModuleList([MGNProcessor(hidden) for _ in range(n_layers)])
        self.flow_dec    = mlp(hidden, flow_out)
        self.heat_dec    = mlp(hidden, heat_out)
        self.species_dec = mlp(hidden, species_out)
    def forward(self, x, edge_index, edge_attr):
        x = self.node_enc(x)
        e = self.edge_enc(edge_attr)
        for proc in self.processors:
            x, e = proc(x, edge_index, e)
        return self.flow_dec(x), self.heat_dec(x), self.species_dec(x)

ckpt      = torch.load(MULTIHEAD_PT, map_location='cpu')
cfg       = ckpt['cfg']
norm      = ckpt['norm']
node_mean = np.array(norm['node_mean'], dtype=np.float32)
node_std  = np.array(norm['node_std'],  dtype=np.float32)
out_mean  = np.array(norm['out_mean'],  dtype=np.float32)
out_std   = np.array(norm['out_std'],   dtype=np.float32)

model = MultiHeadMGN(
    node_dim=cfg['node_input_dim'], edge_dim=cfg['edge_input_dim'],
    flow_out=cfg['flow_out_dim'], heat_out=cfg['heat_out_dim'],
    species_out=cfg['species_out_dim'], hidden=cfg['hidden_dim'], n_layers=cfg['n_layers'],
).to(DEVICE)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Surrogate loaded ({sum(p.numel() for p in model.parameters())/1e6:.2f}M params)')

## 3. Compute objectives for all 80 cases

For each case, run the surrogate and compute:
- `TMA_UI` — TMA uniformity index at standoff mid-plane (z = 5–15 mm)
- `T_UI`   — Temperature uniformity index at wafer (z < 3 mm)
- `neg_Eu` — Negative Euler number (so all objectives are *maximised*)

In [ ]:
K = cfg['k_neighbors']

def uniformity_index(field: np.ndarray) -> float:
    m = float(np.mean(field))
    if abs(m) < 1e-10:
        return float('nan')
    return float(1.0 - np.std(field) / abs(m))


def compute_objectives(h5_path: Path, debug: bool = False) -> dict | None:
    with h5py.File(h5_path, 'r') as f:
        coords       = f['coords'][:].astype(np.float32)
        node_feats   = f['inputs/node_features'][:].astype(np.float32)
        global_feats = f['inputs/global'][:].astype(np.float32)

    N = len(coords)
    gd = dict(zip(GLOBAL_KEYS, global_feats))

    # Build graph
    gb = np.tile(global_feats, (N, 1))
    xi = np.concatenate([node_feats, gb], axis=1)
    tree = cKDTree(coords)
    _, idx = tree.query(coords, k=K+1)
    idx = idx[:, 1:]
    src = np.repeat(np.arange(N), K)
    dst = idx.flatten()
    diff = coords[dst] - coords[src]
    dist = np.linalg.norm(diff, axis=1, keepdims=True)
    med  = float(np.median(dist)) + 1e-8
    ef   = np.concatenate([diff/med, dist/med], axis=1).astype(np.float32)

    x  = torch.from_numpy((xi - node_mean) / node_std).to(DEVICE)
    ei = torch.tensor(np.stack([src, dst]), dtype=torch.long).to(DEVICE)
    ea = torch.from_numpy(ef).to(DEVICE)

    with torch.no_grad():
        fp, hp, sp = model(x, ei, ea)
        fp = fp.cpu().numpy()
        hp = hp.cpu().numpy().flatten()
        sp = sp.cpu().numpy().flatten()
    del x, ei, ea; torch.cuda.empty_cache()

    p_pred   = fp[:, 3] * out_std[3] + out_mean[3]
    T_pred   = hp * out_std[4] + out_mean[4]
    TMA_pred = sp * out_std[5] + out_mean[5]

    z     = coords[:, 2]
    z_min = float(z.min())
    z_max = float(z.max())
    z_rng = z_max - z_min

    if debug:
        print(f'  z range: {z_min:.4f} – {z_max:.4f} m  (span {z_rng*1e3:.1f} mm)')
        print(f'  T_pred:  {T_pred.min():.2f} – {T_pred.max():.2f} K  (mean {T_pred.mean():.2f})')
        print(f'  p_pred:  {p_pred.min():.2f} – {p_pred.max():.2f} Pa')
        print(f'  TMA_pred:{TMA_pred.min():.4e} – {TMA_pred.max():.4e}')

    # Adaptive thresholds: bottom 10% of domain for wafer, 20–60% for standoff mid
    wafer_thresh   = z_min + 0.10 * z_rng
    standoff_lo    = z_min + 0.15 * z_rng
    standoff_hi    = z_min + 0.55 * z_rng

    # T uniformity at wafer region
    wm = z < wafer_thresh
    T_UI = uniformity_index(T_pred[wm]) if wm.sum() > 5 else float('nan')

    # TMA uniformity at standoff mid region
    sm = (z > standoff_lo) & (z < standoff_hi)
    TMA_vals = TMA_pred[sm]
    if sm.sum() > 5 and TMA_vals.max() > 1e-6:
        TMA_UI = uniformity_index(TMA_vals)
    elif sm.sum() > 5:
        TMA_UI = float(-np.std(TMA_vals))   # fallback: -std (maximise = minimise spread)
    else:
        TMA_UI = float('nan')

    # Euler number — cap at physically reasonable range
    delta_p = float(abs(p_pred.max() - p_pred.min()))
    Q_m3s   = float(gd['flow_rate_slm']) * 1.667e-5
    D       = float(gd['D'])
    n_h     = max(int(round(float(gd['n_holes']))), 1)
    V_noz   = Q_m3s / (n_h * np.pi * (D/2)**2)
    Eu = euler(max(delta_p, 1e-3), RHO_N2, max(V_noz, 1e-3))

    return {
        'name':           h5_path.stem,
        'D':              float(gd['D']),
        'pitch_over_D':   float(gd['pitch_over_D']),
        'flow_rate_slm':  float(gd['flow_rate_slm']),
        'Re':             float(gd['Re']),
        'TMA_UI':         TMA_UI,
        'T_UI':           T_UI,
        'neg_Eu':         -Eu,
        'Eu':             Eu,
    }


# ── Debug first case to verify thresholds ──────────────────────────────────
print('=== Debug first case ===')
_ = compute_objectives(h5_files[0], debug=True)
print()

# ── Batch run ───────────────────────────────────────────────────────────────
print(f'Computing objectives for {len(h5_files)} cases...')
obj_records = []
for p in tqdm(h5_files):
    try:
        r = compute_objectives(p)
        if r is not None:
            obj_records.append(r)
    except Exception as e:
        print(f'  skip {p.name}: {e}')

print(f'\nComputed {len(obj_records)} cases')

OBJECTIVE_KEYS = ['TMA_UI', 'T_UI', 'neg_Eu']
obj_valid = {}
for key in OBJECTIVE_KEYS:
    vals    = [r[key] for r in obj_records]
    non_nan = [v for v in vals if not np.isnan(v)]
    obj_valid[key] = non_nan
    rng = f'  [{min(non_nan):.4f}, {max(non_nan):.4f}]  mean {np.mean(non_nan):.4f}' if non_nan else '  ALL NaN'
    print(f'  {key:10s}: {len(non_nan):3d}/80 valid{rng}')

USE_OBJECTIVES = [k for k in OBJECTIVE_KEYS if len(obj_valid[k]) >= 5]
print(f'\nObjectives for BO: {USE_OBJECTIVES}')
if len(USE_OBJECTIVES) < 2:
    print('WARNING: fewer than 2 objectives — BO will be single-objective')

## 4. Build BoTorch dataset

Input `X`: `[log(D/D_min), pitch_over_D, log(Q/Q_min)]` — log-transforms linearise the parameter spaces.  
Output `Y`: `[TMA_UI, T_UI, neg_Eu]` — all maximised.

In [ ]:
# Drop rows where ANY active objective is NaN
clean = [r for r in obj_records
         if not any(np.isnan(r[k]) for k in USE_OBJECTIVES)]
print(f'Clean records (no NaN in active objectives): {len(clean)} / {len(obj_records)}')
print(f'Active objectives: {USE_OBJECTIVES}')

D_min = 0.001; Q_min = 1.0

X_raw = np.array([
    [np.log(r['D'] / D_min),
     r['pitch_over_D'],
     np.log(r['flow_rate_slm'] / Q_min)]
    for r in clean
], dtype=np.float64)   # [N, 3]

Y_raw = np.array(
    [[r[k] for k in USE_OBJECTIVES] for r in clean],
    dtype=np.float64
)   # [N, n_obj]

# Normalise X to [0, 1]
X_lo = X_raw.min(0); X_hi = X_raw.max(0)
X_norm = (X_raw - X_lo) / np.clip(X_hi - X_lo, 1e-8, None)

# Standardise Y
Y_mean = Y_raw.mean(0); Y_std = np.clip(Y_raw.std(0), 1e-8, None)
Y_norm = (Y_raw - Y_mean) / Y_std

X = torch.tensor(X_norm, dtype=torch.float64)
Y = torch.tensor(Y_norm, dtype=torch.float64)

print(f'X: {tuple(X.shape)}  [log(D/D_min), pitch/D, log(Q/Q_min)]')
print(f'Y: {tuple(Y.shape)}  {USE_OBJECTIVES}')

ref_point = torch.tensor(Y_norm.min(0) - 0.1, dtype=torch.float64)
print(f'Reference point: {ref_point.numpy().round(3)}')

## 5. Fit BoTorch GPs (one per objective)

In [ ]:
from botorch.models import SingleTaskGP, ModelListGP
from botorch.fit import fit_gpytorch_mll
from botorch.utils.multi_objective.pareto import is_non_dominated
from gpytorch.mlls import ExactMarginalLogLikelihood
import warnings
warnings.filterwarnings('ignore')

GP_DEVICE = torch.device('cpu')
Xc = X.to(GP_DEVICE)
Yc = Y.to(GP_DEVICE)

n_obj = len(USE_OBJECTIVES)
models = []
for i, obj_name in enumerate(USE_OBJECTIVES):
    gp  = SingleTaskGP(Xc, Yc[:, i:i+1])
    mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
    fit_gpytorch_mll(mll)
    models.append(gp)
    # lengthscale location differs by BoTorch version — try both
    try:
        ls = gp.covar_module.base_kernel.lengthscale.detach().numpy().round(3)
    except AttributeError:
        ls = gp.covar_module.lengthscale.detach().numpy().round(3)
    print(f'  GP [{obj_name}] fitted  lengthscales={ls}')

model_list = ModelListGP(*models)
print(f'GPs ready ({n_obj} objectives).')

## 6. Run qLogNEHVI multi-objective Bayesian optimisation

Runs 5 BO rounds × 4 candidates each = up to 20 new design suggestions.

In [ ]:
from botorch.acquisition.multi_objective.logei import qLogNoisyExpectedHypervolumeImprovement
from botorch.optim import optimize_acqf
from botorch.utils.sampling import sample_simplex
import warnings
warnings.filterwarnings('ignore')

N_ROUNDS    = 5
BATCH_SIZE  = 4
N_RESTARTS  = 10
RAW_SAMPLES = 256

bounds = torch.zeros(2, 3, dtype=torch.float64)
bounds[1] = 1.0   # all dims normalised to [0, 1]

# Accumulate training data across rounds (phantom evaluation via GP mean)
X_train = Xc.clone()
Y_train = Yc.clone()
all_candidates = []

for rnd in range(N_ROUNDS):
    # Refit GPs on current data
    models_rnd = []
    for i in range(n_obj):
        gp  = SingleTaskGP(X_train, Y_train[:, i:i+1])
        mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
        fit_gpytorch_mll(mll)
        models_rnd.append(gp)
    model_rnd = ModelListGP(*models_rnd)

    acqf = qLogNoisyExpectedHypervolumeImprovement(
        model          = model_rnd,
        ref_point      = ref_point,
        X_baseline     = X_train,
        prune_baseline = True,
    )

    candidates, acq_val = optimize_acqf(
        acq_function = acqf,
        bounds       = bounds,
        q            = BATCH_SIZE,
        num_restarts = N_RESTARTS,
        raw_samples  = RAW_SAMPLES,
    )

    # Phantom evaluation: use GP posterior mean as the "observed" value
    with torch.no_grad():
        y_phantom = torch.cat([
            models_rnd[i].posterior(candidates).mean for i in range(n_obj)
        ], dim=-1)   # [q, n_obj]

    X_train = torch.cat([X_train, candidates], dim=0)
    Y_train = torch.cat([Y_train, y_phantom], dim=0)
    all_candidates.append(candidates.numpy())

    print(f'Round {rnd+1}/{N_ROUNDS}  acq={acq_val:.4f}  '
          f'pool_size={len(X_train)}')

print('\nBO complete.')

## 7. Pareto front analysis

In [ ]:
import matplotlib.pyplot as plt

X_all = X_train.numpy()
Y_all = Y_train.numpy()

pareto_mask = is_non_dominated(torch.tensor(Y_all)).numpy()
print(f'Pareto front: {pareto_mask.sum()} / {len(Y_all)} points')

Y_disp = Y_all * Y_std + Y_mean   # denormalise

n_obj  = len(USE_OBJECTIVES)
n_pairs = min(n_obj * (n_obj - 1) // 2, 3)
fig, axes = plt.subplots(1, max(n_pairs, 1), figsize=(6 * max(n_pairs, 1), 5))
if n_pairs == 1:
    axes = [axes]

pair_idx = 0
for i in range(n_obj):
    for j in range(i+1, n_obj):
        if pair_idx >= len(axes): break
        ax = axes[pair_idx]
        ax.scatter(Y_disp[~pareto_mask, i], Y_disp[~pareto_mask, j],
                   s=20, alpha=0.4, label='All', c='steelblue')
        ax.scatter(Y_disp[pareto_mask, i], Y_disp[pareto_mask, j],
                   s=60, marker='*', c='red', label='Pareto', zorder=5)
        ax.set_xlabel(USE_OBJECTIVES[i]); ax.set_ylabel(USE_OBJECTIVES[j])
        ax.set_title(f'{USE_OBJECTIVES[i]} vs {USE_OBJECTIVES[j]}')
        ax.legend()
        pair_idx += 1

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'pareto_front.png'), dpi=120, bbox_inches='tight')
plt.show()

## 8. Top Pareto candidates — decode back to physical design params

In [ ]:
def decode_x(x_norm: np.ndarray) -> dict:
    x_raw = x_norm * (X_hi - X_lo) + X_lo
    return {
        'D_mm':         float(np.exp(x_raw[0]) * D_min * 1e3),
        'pitch_over_D': float(x_raw[1]),
        'Q_slm':        float(np.exp(x_raw[2]) * Q_min),
    }

pareto_X = X_all[pareto_mask]
pareto_Y = Y_disp[pareto_mask]

# Sort by first objective (TMA_UI if present, else T_UI)
order = np.argsort(-pareto_Y[:, 0])

header = '  '.join(f'{k:>10s}' for k in USE_OBJECTIVES)
print(f'Top Pareto candidates (sorted by {USE_OBJECTIVES[0]}):\n')
print(f'{"Rank":>4}  {"D [mm]":>8}  {"pitch/D":>8}  {"Q [slm]":>8}  {header}')
print('-' * (40 + 12 * len(USE_OBJECTIVES)))

top_candidates = []
for rank, i in enumerate(order[:10]):
    p = decode_x(pareto_X[i])
    vals = pareto_Y[i]
    obj_str = '  '.join(f'{v:>10.4f}' for v in vals)
    print(f'{rank+1:>4}  {p["D_mm"]:>8.2f}  {p["pitch_over_D"]:>8.2f}  {p["Q_slm"]:>8.2f}  {obj_str}')
    top_candidates.append({
        **p,
        **{k: float(v) for k, v in zip(USE_OBJECTIVES, vals)},
        'pareto_rank': rank + 1,
    })

## 9. Save results

In [ ]:
def _to_py(obj):
    if isinstance(obj, (np.floating, np.integer)): return float(obj)
    if isinstance(obj, dict):  return {k: _to_py(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [_to_py(i) for i in obj]
    return obj

output = {
    'n_observed':        len(obj_records),
    'n_pareto':          int(pareto_mask.sum()),
    'top_candidates':    _to_py(top_candidates),
    'objectives_all':    _to_py([{**r} for r in obj_records]),
}

out_path = OUT_DIR / 'optimizer_results.json'
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Saved → {out_path}')

# Final summary
top = top_candidates[0]
eu_val  = -top.get('neg_Eu', float('nan'))
tma_val = top.get('TMA_UI', float('nan'))
t_val   = top.get('T_UI',   float('nan'))

print(f'\nOptimizer summary:')
print(f'  Cases evaluated: {len(obj_records)}')
print(f'  Pareto front:    {pareto_mask.sum()} designs')
print(f'  BO suggestions:  {N_ROUNDS * BATCH_SIZE} new candidates')
print(f'\nTop candidate:')
print(f'  D = {top["D_mm"]:.2f} mm   pitch/D = {top["pitch_over_D"]:.2f}   '
      f'Q = {top["Q_slm"]:.2f} slm')
print(f'  TMA_UI = {tma_val:.4f}   T_UI = {t_val:.4f}   Eu = {eu_val:.2f}')
print('\nReady for 08_streamlit_app.ipynb')